# Code Explanation
1)Connects to Amazon Bedrock (Knowledge Base/Agent service).

2) Lists knowledge bases in your account (limited to 1).

3) Prints the Knowledge Base ID if found.

4) Otherwise, prints a message that no KBs exist.

5) Handles API errors gracefully.

This is typically the first step in a RAG workflow  confirming that your Bedrock Knowledge Base exists and retrieving its ID.

# Details



* boto3 → AWS SDK for Python, used to interact with AWS services.
* botocore → low-level core library used by boto3, provides error handling, request/response handling.
* Creates a Bedrock Agent client using bedrock-agent.

This is the service namespace for interacting with Amazon Bedrock Knowledge Bases and Agents.


In [ ]:
import boto3
import botocore

session = boto3.Session()
bedrock_client = session.client('bedrock-agent')

try:
    response = bedrock_client.list_knowledge_bases(
        maxResults=1  # We only need to retrieve the first Knowledge Base
    )
    knowledge_base_summaries = response.get('knowledgeBaseSummaries', [])

    if knowledge_base_summaries:
        kb_id = knowledge_base_summaries[0]['knowledgeBaseId']
        print(f"Knowledge Base ID: {kb_id}")
    else:
        print("No Knowledge Base summaries found.")

except botocore.exceptions.ClientError as e:
    print(f"Error: {e}")

#Code Explanation

1. Prepares AWS session & region.
2. Sets up a Bedrock Runtime client (for foundation model inference).
3. Sets up a Bedrock Agent Runtime client (for RAG apps, knowledge base queries, or agent flows).
4. Configures clients with long timeouts & no retries to handle potentially long-running calls.
5. Includes a pretty printer for cleanly displaying responses.

#Details

* boto3 → AWS SDK for Python (interacts with AWS services).
* Config (from botocore.client) → lets you customize client settings like timeouts and retries.
* pprint → pretty printer for nicer JSON/dict printing.
* json → standard Python library for working with JSON.
* bedrock_client → talk to foundation models directly.
* bedrock_agent_client → talk to knowledge bases / RAG agents.


In [ ]:
import boto3
from botocore.client import Config
import pprint
import json

pp = pprint.PrettyPrinter(indent=2)

session = boto3.session.Session()
region = session.region_name

bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})
bedrock_client = boto3.client('bedrock-runtime', region_name = region)
bedrock_agent_client = boto3.client("bedrock-agent-runtime",
                              config=bedrock_config, region_name = region)


#Code Explanation
This function retrieves relevant context from your Knowledge Base using hybrid semantic + keyword search to support your RAG QA pipeline.


In [ ]:
def retrieve(query, kbId, numberOfResults=5):
    return bedrock_agent_client.retrieve(
        retrievalQuery= {
            'text': query
        },
        knowledgeBaseId=kbId,
        retrievalConfiguration= {
            'vectorSearchConfiguration': {
                'numberOfResults': numberOfResults,
                'overrideSearchType': "HYBRID", # optional
            }
        }
    )

# Code Explanation
1. User asks: “What was the total operating lease liabilities and total sublease income … ?”
2. Bedrock Knowledge Base searches across embedded vectors + keywords.
3. Returns top 5 most relevant text chunks from your documents (financial reports in S3, indexed in OpenSearch).
4. You now have retrieved context.
5. Next step in RAG → Pass this retrievalResults into a Bedrock LLM (e.g., Nova Pro) to generate a final answer.


In [ ]:
query = "What was the total operating lease liabilities and total sublease income of the AnyCompany as of December 31, 2022?"
response = retrieve(query, kb_id, 5)
retrievalResults = response['retrievalResults']
pp.pprint(retrievalResults)

#Code Explanation
The function explanation,
1. Take retrieval results → input is the list of results returned from Knowledge Base search.
2. Initialize an empty list → contexts = [].
3. Loop through results → for each retrieved result (a dictionary with text + metadata + score).
4. Extract only the text → retrievedResult['content']['text'].
5. Append text to list → collect all chunks into contexts.
6. Return the list of texts → clean context chunks ready for LLM input.


In [ ]:
# fetch context from the response
def get_contexts(retrievalResults):
    contexts = []
    for retrievedResult in retrievalResults:
        contexts.append(retrievedResult['content']['text'])
    return contexts

#Code Explanation

This step verifies what knowledge (context text) has been retrieved from your S3 documents (via the Bedrock Knowledge Base + OpenSearch).
Later, these contexts will be fed into the LLM (e.g., Nova Pro/Lite) as grounding context for the RAG answer.


In [ ]:
contexts = get_contexts(retrievalResults)
pp.pprint(contexts)

#Code Explanation

This is the prompt engineering step of your RAG pipeline.

This prompt construction step is where you merge:

🡪 User Query (query)

🡪 Retrieved Context (contexts)

🡪 System Instructions (rules & guidelines)

The result is a structured, context-grounded prompt that can be safely sent to the Bedrock LLM (bedrock-runtime) for generation.

In [ ]:
prompt = f"""
Human: You are a financial advisor AI system specializing in analyzing financial statements, particularly focusing on lease accounting and disclosures. Use the following pieces of information to provide a detailed answer to the question enclosed in <question> tags.

When analyzing the information, please follow these guidelines:
1. Look for specific details about operating lease liabilities and sublease income as of December 31, 2022.
2. If exact figures for the requested date are not available, provide the most recent available data and clearly state the date.
3. Include any relevant information about lease terms, weighted-average remaining lease terms, or lease accounting policies.
4. If the information is not explicitly about AnyCompany, but general lease accounting information is provided, summarize that instead.
5. If you don't find specific lease information, mention any other financial data that might be relevant to understanding the company's obligations or income streams.

<context>
{contexts}
</context>

<question>
{query}
</question>

In your response:
- Provide specific numerical figures where available, using proper currency notation (e.g., $X million).
- Clearly indicate what information is directly from the context and what might be inferred.
- If certain information is not available, explicitly state: "The [specific detail] is not provided in the available financial statements."
- Include any other relevant lease-related or financial information found in the context.

Assistant:"""

#Code Explanation

You’re packaging the user’s structured prompt into the format Nova Lite expects, with generation controls, so you can send this payload to bedrock-runtime.invoke_model().


In [ ]:
# payload with model parameters
messages = [{
    "role": "user",
    "content": [{"text": prompt}]
}]

# Create the proper Nova Lite payload
nova_payload = {
    "schemaVersion": "messages-v1",
    "messages": messages,
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.5,
        "topP": 0.9,
        "topK": 20
    }
}

#Code Explanation
1. Pick model + formats
2. Send payload with invoke_model()
3. Parse JSON response
4. Extract just the text reply
5. Print it


In [ ]:
modelId = 'amazon.nova-lite-v1:0' # change this to use a different version from the model provider
accept = 'application/json'
contentType = 'application/json'

response = bedrock_client.invoke_model(
    body=json.dumps(nova_payload),
    modelId=modelId,
    accept=accept,
    contentType=contentType
)

# Parse and extract the response
response_body = json.loads(response.get('body').read())

# Extract just the text from the response
response_text = ''
if 'output' in response_body and 'message' in response_body['output']:
    message_content = response_body['output']['message']['content']
    if message_content and isinstance(message_content, list):
        response_text = message_content[0].get('text', '')

# Print the response text
print(response_text)

# Part 2: Hybrid Search, Reranking, Retrieval Monitoring, Citations and Hallucination Detection

The cells above build a basic RAG pipeline: retrieve chunks from the Knowledge Base, put them into a prompt, and generate an answer with Amazon Nova Lite. The steps below make that pipeline more reliable and easier to check:

| Step | What it adds |
|---|---|
| **Step 1** | Configuration: Region, models and clients |
| **Step 2** | **Hybrid search:** BM25 keyword search + semantic vector search, combined with Reciprocal Rank Fusion |
| **Step 3** | **Reranking:** Cohere Rerank 3.5 re-scores the hybrid search results and keeps the most relevant chunks |
| **Step 4** | **Retrieval monitoring:** every query and its retrieved chunks are logged to a JSON Lines file |
| **Step 5** | **Citation generation:** the answer cites `[S1]`, `[S2]`, ... and returns the source file, page and S3 location of each |
| **Step 6** | **Hallucination detection:** three checks on every answer, combined into a risk level |
| **Step 7** | The complete pipeline, tested with the sample questions from the activity guide |

**Region:**  Run this notebook in **US East (N. Virginia) `us-east-1`**, the Region where you created the Knowledge Base, the IAM policies and the S3 bucket. The new steps always call `us-east-1`.

**Run the cells above first.** The new steps use `kb_id` from the first cell and `query` from the retrieval cell.

## Step 1: Configuration

Install the BM25 library, then set the Region, the models and the clients for the new steps.

In [ ]:
%pip install -q rank-bm25

### Code Explanation

**`rank-bm25`:**  A small Python library that implements BM25, the standard keyword ranking algorithm used by search engines such as OpenSearch. It is used in Step 2. It runs locally and calls no AWS service.

In [ ]:
# Step 1: Configuration for hybrid search, reranking, monitoring, citations and hallucination checks
import os
import re
import json
import time
import uuid
import hashlib
from collections import Counter
from datetime import datetime, timezone
from typing import Dict, List
from urllib.parse import unquote

import boto3
from botocore.client import Config
from botocore.exceptions import ClientError
from rank_bm25 import BM25Okapi

# ------------------------------------------------------------------------------
# Region and models
# ------------------------------------------------------------------------------
AWS_REGION = 'us-east-1'                  # the Region used for every resource in the activity guide
KB_NAME = 'knowledge-base-project-2025'   # the Knowledge Base name from the activity guide
LLM_MODEL_ID = 'amazon.nova-lite-v1:0'    # writes the answers (the same model as the cells above)
JUDGE_MODEL_ID = LLM_MODEL_ID             # checks the answers in Step 6; must be an Amazon Nova model

# Reranker (Step 3). Cohere Rerank 3.5 is available in us-east-1; Amazon Rerank 1.0 is not.
RERANK_MODEL_ID = 'cohere.rerank-v3-5:0'
RERANK_MODEL_ARN = f'arn:aws:bedrock:{AWS_REGION}::foundation-model/{RERANK_MODEL_ID}'
USE_RERANKING = True                      # set to False to skip reranking everywhere

notebook_region = boto3.session.Session().region_name
if notebook_region != AWS_REGION:
    print(f"⚠️ This notebook's default Region is {notebook_region}, but the lab resources are in {AWS_REGION}.")
    print(f"   The new steps below always use {AWS_REGION}.\n")

# Clients pinned to the lab Region. Retries help when Bedrock throttles requests.
retry_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 3, 'mode': 'standard'})
bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION, config=retry_config)
kb_runtime = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION, config=retry_config)

# Use the Knowledge Base ID found in the first cell, or look it up by name
if 'kb_id' not in globals():
    kb_list = boto3.client('bedrock-agent', region_name=AWS_REGION).list_knowledge_bases(maxResults=50)
    summaries = kb_list.get('knowledgeBaseSummaries', [])
    matches = [s for s in summaries if s['name'] == KB_NAME] or summaries
    if not matches:
        raise RuntimeError(f"No Knowledge Base found in {AWS_REGION}. Complete the Knowledge Base tasks in the guide first.")
    kb_id = matches[0]['knowledgeBaseId']


def explain_bedrock_error(error: Exception) -> str:
    """Turn an AWS error into a short message with a hint on how to fix it."""
    if isinstance(error, ClientError):
        code = error.response['Error']['Code']
        message = error.response['Error']['Message']
        hints = {
            'AccessDeniedException': 'Check that the SageMaker execution role has AmazonBedrockFullAccess '
                                     'and that you can use this model in this Region.',
            'ResourceNotFoundException': f'Check that the Knowledge Base exists in {AWS_REGION} and that kb_id is correct.',
            'ValidationException': f'Check the model ID and the request. The model must be available in {AWS_REGION}.',
            'ThrottlingException': 'Too many requests. Wait a few seconds and run the cell again.',
        }
        return f"{code}: {message}" + (f" → {hints[code]}" if code in hints else '')
    return f"{type(error).__name__}: {error}"


print("✅ Configuration ready")
print(f"   Region:          {AWS_REGION}")
print(f"   Knowledge Base:  {kb_id}")
print(f"   Answer model:    {LLM_MODEL_ID}")
print(f"   Judge model:     {JUDGE_MODEL_ID}")
print(f"   Reranker:        {RERANK_MODEL_ID}" + ("" if USE_RERANKING else " (switched off)"))

### Code Explanation

**Region:**  `AWS_REGION = 'us-east-1'` pins the new clients to the lab Region. The cells above use the notebook's default Region; if that is not `us-east-1`, the cell prints a warning.

**Models:**
- `LLM_MODEL_ID` (`amazon.nova-lite-v1:0`) writes the answers. It is the same model the cells above use, and the one the activity guide gives access to.
- `JUDGE_MODEL_ID` checks the answers in Step 6. By default it is the same Nova Lite model, which keeps the lab cheap. A model checking its own kind of output can miss its own mistakes, so for production use a stronger judge such as Amazon Nova Pro (`amazon.nova-pro-v1:0`), if your account can use it. The judge must be an Amazon Nova model, because `invoke_llm()` uses the Nova request format.
- `RERANK_MODEL_ID` (`cohere.rerank-v3-5:0`) reorders the retrieved chunks in Step 3. Use Cohere Rerank 3.5 in `us-east-1`: Amazon Rerank 1.0 is not available in this Region, which is a common reason for a reranking cell to fail. `RERANK_MODEL_ARN` is the model ARN the Rerank API needs. Set `USE_RERANKING = False` to switch reranking off in every later step.
- The embedding model (Titan Text Embeddings V2) is set in the Knowledge Base itself and is not changed here. BM25 needs no model.

**Clients:**  `bedrock_runtime` calls the models and `kb_runtime` calls the Knowledge Base Retrieve API. They have new names so that they do not replace `bedrock_client` above, and they retry up to 3 times when Bedrock throttles requests.

**`kb_id`:**  Taken from the first cell. If that cell did not set it, the Knowledge Base is looked up by its name, `knowledge-base-project-2025`.

**`explain_bedrock_error()`:**  Turns an AWS error into a short message with a hint, for example to check the IAM role or the Region.

## Step 2: Hybrid Search (BM25 Keyword Search + Semantic Vector Search)

Retrieve candidate chunks from the Knowledge Base, rank them with both vector search and BM25 keyword search, and combine the two rankings.

In [ ]:
# Step 2: Hybrid search (BM25 keyword search + semantic vector search)

# Common words that carry no meaning for keyword matching
STOP_WORDS = {
    'a', 'an', 'the', 'and', 'or', 'of', 'to', 'in', 'on', 'for', 'by', 'with', 'as', 'at', 'from',
    'is', 'are', 'was', 'were', 'be', 'been', 'it', 'its', 'this', 'that', 'these', 'those',
    'what', 'which', 'who', 'how', 'why', 'when', 'where', 'does', 'do', 'did', 'can', 'could',
    'should', 'would', 'will', 'has', 'have', 'had', 'their', 'there', 'about', 'any', 'all'
}
RRF_K = 60   # Reciprocal Rank Fusion constant (value from the original RRF paper)


def tokenize(text: str) -> List[str]:
    """Lower-case the text, split it into words and remove stop words."""
    return [w for w in re.findall(r'[a-z0-9]+', text.lower()) if w not in STOP_WORDS]


def chunk_location(item: Dict) -> Dict:
    """Return the file name, S3 URI, page number and chunk ID of one Retrieve result."""
    metadata = item.get('metadata') or {}
    uri = (item.get('location', {}).get('s3Location', {}).get('uri')
           or metadata.get('x-amz-bedrock-kb-source-uri') or 'unknown')
    page = metadata.get('x-amz-bedrock-kb-document-page-number')
    chunk_id = metadata.get('x-amz-bedrock-kb-chunk-id')
    if not chunk_id:   # fall back to a fingerprint of the text
        chunk_id = 'text-' + hashlib.md5(item['content']['text'].encode('utf-8')).hexdigest()[:12]
    return {
        'file': uri.rsplit('/', 1)[-1],
        'uri': uri,
        'page': int(page) if page is not None else None,
        'chunk_id': unquote(str(chunk_id))
    }


def kb_retrieve(query: str, search_type: str = 'HYBRID', number_of_results: int = 10) -> List[Dict]:
    """Call the Knowledge Base Retrieve API with SEMANTIC (vector only) or HYBRID search."""
    response = kb_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'vectorSearchConfiguration': {
                'numberOfResults': number_of_results,
                'overrideSearchType': search_type
            }
        }
    )
    results = []
    for rank, item in enumerate(response['retrievalResults'], 1):
        results.append({'rank': rank, 'text': item['content']['text'], 'score': item.get('score'),
                        **chunk_location(item)})
    return results


def hybrid_search(query: str, k: int = 5, candidates: int = 10) -> List[Dict]:
    """Combine semantic vector search and BM25 keyword search with Reciprocal Rank Fusion."""
    # 1. Collect candidate chunks: vector search, plus Bedrock's own hybrid search,
    #    which adds chunks that match the query words but not its meaning
    semantic = kb_retrieve(query, 'SEMANTIC', candidates)
    kb_hybrid = kb_retrieve(query, 'HYBRID', candidates)

    pool = {}
    for r in semantic + kb_hybrid:
        pool.setdefault(r['chunk_id'], r)
    if not pool:
        return []
    chunk_ids = list(pool)

    vector_rank = {r['chunk_id']: r['rank'] for r in semantic}
    vector_score = {r['chunk_id']: r['score'] for r in semantic}
    kb_hybrid_rank = {r['chunk_id']: r['rank'] for r in kb_hybrid}

    # 2. Keyword search: score every candidate with BM25 and keep the best `candidates`
    bm25 = BM25Okapi([tokenize(pool[cid]['text']) for cid in chunk_ids])
    scores = bm25.get_scores(tokenize(query))
    bm25_score = {cid: float(s) for cid, s in zip(chunk_ids, scores)}
    keyword_hits = sorted((cid for cid in chunk_ids if bm25_score[cid] > 0),
                          key=lambda cid: bm25_score[cid], reverse=True)[:candidates]
    bm25_rank = {cid: rank for rank, cid in enumerate(keyword_hits, 1)}

    # 3. Reciprocal Rank Fusion: every list that found a chunk adds 1 / (RRF_K + rank)
    fused = []
    for cid in chunk_ids:
        in_vector, in_keyword = cid in vector_rank, cid in bm25_rank
        if not (in_vector or in_keyword):
            continue
        rrf = (1 / (RRF_K + vector_rank[cid]) if in_vector else 0) + \
              (1 / (RRF_K + bm25_rank[cid]) if in_keyword else 0)
        fused.append({
            **{key: pool[cid][key] for key in ('text', 'file', 'uri', 'page', 'chunk_id')},
            'found_by': 'both' if in_vector and in_keyword else ('vector' if in_vector else 'keyword'),
            'rrf_score': rrf,
            'vector_rank': vector_rank.get(cid),
            'vector_score': vector_score.get(cid),
            'bm25_rank': bm25_rank.get(cid),
            'bm25_score': bm25_score[cid],
            'kb_hybrid_rank': kb_hybrid_rank.get(cid)
        })

    fused.sort(key=lambda r: r['rrf_score'], reverse=True)
    for rank, r in enumerate(fused[:k], 1):
        r['rank'] = rank
    return fused[:k]


def print_hybrid_results(query: str, results: List[Dict]):
    print(f"\n🔎 {query}")
    print(f"   {'#':<3}{'found by':<10}{'vector':<8}{'bm25':<6}{'page':<6}text")
    for r in results:
        print(f"   {r['rank']:<3}{r['found_by']:<10}{r['vector_rank'] or '-':<8}{r['bm25_rank'] or '-':<6}"
              f"{r['page'] or '-':<6}{r['text'][:70].replace(chr(10), ' ')}...")


# Test: compare vector-only search with hybrid search
print("=" * 90)
print("🧪 TESTING HYBRID SEARCH")
print("=" * 90)
test_queries = [
    globals().get('query') or "What was the total operating lease liabilities of AnyCompany?",  # lease question from above
    "sublease income",                                                   # exact financial term
    "How does the company protect itself against currency movements?"   # meaning, not exact words
]
for test_query in test_queries:
    try:
        results = hybrid_search(test_query, k=5)
        semantic_ids = {r['chunk_id'] for r in kb_retrieve(test_query, 'SEMANTIC', 5)}
        new_chunks = sum(1 for r in results if r['chunk_id'] not in semantic_ids)
        print_hybrid_results(test_query, results)
        print(f"   ➕ {new_chunks} of {len(results)} chunks are not in the vector-only top 5")
    except Exception as e:
        print(f"❌ {test_query}: {explain_bedrock_error(e)}")

### Code Explanation

**Why Hybrid Search:**  The two kinds of search fail in different ways.
- **Semantic vector search** matches meaning. It finds a chunk about "foreign exchange rates" for a question about "currency movements", but it can miss exact terms, names and figures.
- **Keyword search (BM25)** matches the words themselves. It reliably finds "sublease income" or "operating lease liabilities", but it cannot connect words that mean the same thing.

Hybrid search runs both and merges the results, so each covers the other's blind spot. This matters for financial documents, which are full of exact terms.

**`kb_retrieve()`:**  Calls the Retrieve API with `overrideSearchType` set to `SEMANTIC` (vector search only) or `HYBRID` (Bedrock's built-in keyword + vector search). Each result keeps its text, relevance score, file name, S3 URI, page number and chunk ID, taken from the result's `location` and `metadata`. These are used for citations in Step 5.

**`hybrid_search()`:**
1. **Candidates:** retrieves the top `candidates` chunks (default 10) with `SEMANTIC` search and with `HYBRID` search, and merges them. The `HYBRID` call adds chunks that OpenSearch found by keyword but vector search missed.
2. **Vector ranking:** the order of the `SEMANTIC` results.
3. **Keyword ranking:** `BM25Okapi` scores every candidate by how often the query words appear in it, giving more weight to rare words. `tokenize()` lower-cases the text and removes common words such as "the" and "what".
4. **Reciprocal Rank Fusion (RRF):** vector scores and BM25 scores use different scales, so they cannot be added. RRF uses only the rank: a chunk earns `1 / (60 + rank)` from every ranking that found it. A chunk ranked high by both gets the highest total.

Each result has a `found_by` field: `both`, `vector` or `keyword`, plus the ranks and scores used. Step 4 logs them.

**Note:**  BM25 here scores the candidate chunks, not the whole Knowledge Base, because the Retrieve API does not return all chunks. The corpus-wide keyword search is done by the `HYBRID` call, which uses OpenSearch's own keyword index. Raise `candidates` (up to 100) to give BM25 more chunks to rank.

**Test:**  Three queries. The line `➕ N of 5 chunks are not in the vector-only top 5` shows how many chunks hybrid search added that vector search alone would not have returned. Expect more of them for the exact term `sublease income`.

**Cost:**  Each hybrid search makes two Retrieve calls. BM25 runs locally and is free.

## Step 3: Reranking with Cohere Rerank 3.5

Retrieve a larger set of chunks with hybrid search, then let a reranking model re-score them against the question and keep only the most relevant ones.

In [ ]:
# Step 3: Reranking with Cohere Rerank 3.5 (re-score the hybrid search results)

def rerank_results(query: str, results: List[Dict], top_n: int = 5):
    """Re-score retrieved chunks with the reranker and return the best `top_n`.
    If the reranker cannot be called, the hybrid search order is kept, so the pipeline still runs."""
    for hybrid_rank, r in enumerate(results, 1):
        r['hybrid_rank'] = hybrid_rank
        r['rerank_score'] = None
    if not results:
        return [], {'reranked': False, 'latency_ms': 0, 'error': None}

    start = time.time()
    try:
        response = kb_runtime.rerank(
            queries=[{'type': 'TEXT', 'textQuery': {'text': query}}],
            sources=[
                {'type': 'INLINE',
                 'inlineDocumentSource': {'type': 'TEXT', 'textDocument': {'text': r['text']}}}
                for r in results
            ],
            rerankingConfiguration={
                'type': 'BEDROCK_RERANKING_MODEL',
                'bedrockRerankingConfiguration': {
                    'numberOfResults': min(top_n, len(results)),
                    'modelConfiguration': {'modelArn': RERANK_MODEL_ARN}
                }
            }
        )
    except Exception as e:
        info = {'reranked': False, 'latency_ms': round((time.time() - start) * 1000),
                'error': explain_bedrock_error(e)}
        fallback = results[:top_n]
        for rank, r in enumerate(fallback, 1):
            r['rank'] = rank
        return fallback, info

    reranked = []
    for rank, item in enumerate(response['results'], 1):    # results come back best first
        r = results[item['index']]                          # 'index' points into the list we sent
        r['rerank_score'] = item['relevanceScore']
        r['rank'] = rank
        reranked.append(r)
    info = {'reranked': True, 'latency_ms': round((time.time() - start) * 1000), 'error': None}
    return reranked, info


def _without_reranking(results: List[Dict]):
    for hybrid_rank, r in enumerate(results, 1):
        r['hybrid_rank'], r['rerank_score'] = hybrid_rank, None
    return results, {'reranked': False, 'latency_ms': 0, 'error': None, 'skipped': True}


def search_and_rerank(query: str, k: int = 5, candidates: int = 10, rerank_pool: int = 15,
                      rerank: bool = None):
    """Hybrid search for `rerank_pool` chunks, then keep the `k` best according to the reranker."""
    rerank = USE_RERANKING if rerank is None else rerank
    if not rerank:
        results = hybrid_search(query, k=k, candidates=candidates)
        return _without_reranking(results)
    pool = hybrid_search(query, k=rerank_pool, candidates=candidates)
    return rerank_results(query, pool, top_n=k)


def print_reranked(query: str, results: List[Dict], info: Dict, k: int):
    print(f"\n🔎 {query}")
    if info.get('error'):
        print(f"   ⚠️ Reranking failed, using the hybrid search order instead.\n      {info['error']}")
        if 'AccessDenied' in info['error'] or 'ValidationException' in info['error']:
            print(f"      In the Amazon Bedrock console ({AWS_REGION}), open the model catalog, find "
                  f"Cohere Rerank 3.5 and make sure your account can use it. Then run this cell again.")
    else:
        print(f"   Reranked in {info['latency_ms']} ms")
    print(f"   {'new':<5}{'hybrid':<8}{'rerank score':<14}{'page':<6}text")
    for r in results:
        score = f"{r['rerank_score']:.3f}" if r['rerank_score'] is not None else '-'
        print(f"   {r['rank']:<5}{r['hybrid_rank']:<8}{score:<14}{r['page'] or '-':<6}"
              f"{r['text'][:65].replace(chr(10), ' ')}...")
    promoted = sum(1 for r in results if r['hybrid_rank'] > k)
    if info.get('reranked'):
        print(f"   ⬆️ {promoted} of {len(results)} chunks were outside the hybrid top {k} and were moved in by the reranker")


# Test: compare the hybrid order with the reranked order
print("=" * 90)
print(f"🧪 TESTING RERANKING ({RERANK_MODEL_ID})")
print("=" * 90)
for test_query in [
    globals().get('query') or "What was the total operating lease liabilities of AnyCompany?",
    "How does AnyCompany Financial mitigate the risks associated with changes in commodity prices?"
]:
    try:
        results, info = search_and_rerank(test_query, k=5, rerank=True)
        print_reranked(test_query, results, info, k=5)
    except Exception as e:
        print(f"❌ {test_query}: {explain_bedrock_error(e)}")

### Code Explanation

**Why Rerank:**  Vector search and BM25 are fast because they score the question and each chunk separately. A **reranker** is a model that reads the question and a chunk *together* and scores how well the chunk answers the question. It is slower, so it is used only on a short list. The usual pattern is: retrieve a wide set of candidates cheaply (Step 2), then rerank them and pass only the best few to the LLM. Better chunks in the prompt mean better answers and fewer hallucinations.

**Model:**  Cohere Rerank 3.5 (`cohere.rerank-v3-5:0`), called through the Amazon Bedrock **Rerank API** of the `bedrock-agent-runtime` client (the same client used for the Retrieve API). It is available in `us-east-1`. Amazon Rerank 1.0 is **not** available in `us-east-1`, so do not switch to it in this lab.

**`rerank_results()`:**
- Sends the question and every chunk text to `kb_runtime.rerank()`. Each chunk is an `INLINE` text source.
- `numberOfResults` keeps only the best `top_n` chunks. The reply lists them best first, each with the `index` of the chunk in the list that was sent and a `relevanceScore` between 0 and 1.
- Every chunk keeps its position from Step 2 as `hybrid_rank`, and gets its new `rank` and its `rerank_score`.
- **Fallback:** if the Rerank call fails (for example, no access to the model, or a different Region), the function keeps the hybrid search order and returns the error message instead of stopping the notebook. Step 4 logs the failure.

**`search_and_rerank()`:**  Runs `hybrid_search()` for `rerank_pool` chunks (default 15), then reranks them and keeps the best `k` (default 5). With `rerank=False`, it returns the hybrid search results without reranking. By default it follows `USE_RERANKING` from Step 1.

**Test:**  For two questions, the table shows each chunk's new rank, its rank before reranking (`hybrid`) and its rerank score. The line `⬆️ N of 5 chunks were outside the hybrid top 5` shows how many chunks the reranker brought in that hybrid search alone would have left out.

**If the cell shows `⚠️ Reranking failed`:**
- `AccessDeniedException` or a message about model access: in the Amazon Bedrock console in `us-east-1`, open the model catalog, find **Cohere Rerank 3.5** and make sure your account can use it. Some accounts must accept the model's terms once before the first call.
- `ValidationException` about the model: check that `AWS_REGION` is `us-east-1` and `RERANK_MODEL_ID` is `cohere.rerank-v3-5:0`.
- The rest of the notebook still runs, using the hybrid search order.

**Permissions:**  The Rerank API needs `bedrock:Rerank` and `bedrock:InvokeModel` on the reranking model. The `AmazonBedrockFullAccess` policy on your SageMaker execution role includes both.

**Cost:**  One Rerank call per question, billed per query. See Amazon Bedrock pricing for the current rate.

## Step 4: Retrieval Pipeline Monitoring

Log every query and the chunks retrieved for it to a JSON Lines file, then summarize the log.

In [ ]:
# Step 4: Retrieval pipeline monitoring (log the retrieved and reranked chunks for every query)

# Every retrieval (and, from Step 7, every generated answer) is written as one JSON line
LOG_DIR = 'logs'
RAG_LOG_PATH = f'{LOG_DIR}/rag_log.jsonl'
os.makedirs(LOG_DIR, exist_ok=True)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def write_log_event(event: Dict):
    """Append one event to the JSON Lines log file."""
    with open(RAG_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(event, ensure_ascii=False, default=str) + '\n')


def read_log_events(event_type: str = None) -> List[Dict]:
    """Read all events from the log, optionally only one type ('retrieval' or 'generation')."""
    if not os.path.exists(RAG_LOG_PATH):
        return []
    with open(RAG_LOG_PATH, 'r', encoding='utf-8') as f:
        events = [json.loads(line) for line in f if line.strip()]
    if event_type:
        events = [e for e in events if e.get('event') == event_type]
    return events


def monitored_search(query: str, k: int = 5, candidates: int = 10, rerank: bool = None):
    """Run hybrid search + reranking and log the query, the retrieved chunks and the latency."""
    rerank = USE_RERANKING if rerank is None else rerank
    query_id = uuid.uuid4().hex[:8]
    start = time.time()
    error = None
    rerank_info = {'reranked': False, 'latency_ms': 0, 'error': None}
    try:
        results, rerank_info = search_and_rerank(query, k=k, candidates=candidates, rerank=rerank)
    except Exception as e:
        results, error = [], explain_bedrock_error(e)
    latency_ms = round((time.time() - start) * 1000)

    event = {
        'event': 'retrieval',
        'query_id': query_id,
        'timestamp': utc_now(),
        'knowledge_base_id': kb_id,
        'query': query,
        'method': 'hybrid_bm25_vector_rrf' + ('+rerank' if rerank_info['reranked'] else ''),
        'k': k,
        'candidates': candidates,
        'latency_ms': latency_ms,
        'rerank_requested': rerank,
        'reranked': rerank_info['reranked'],
        'rerank_model': RERANK_MODEL_ID if rerank else None,
        'rerank_latency_ms': rerank_info['latency_ms'],
        'rerank_error': rerank_info['error'],
        'num_chunks': len(results),
        'chunks': [
            {
                'rank': r['rank'],
                'hybrid_rank': r.get('hybrid_rank'),
                'rerank_score': round(r['rerank_score'], 4) if r.get('rerank_score') is not None else None,
                'chunk_id': r['chunk_id'],
                'file': r['file'],
                'page': r['page'],
                'found_by': r['found_by'],
                'rrf_score': round(r['rrf_score'], 5),
                'vector_rank': r['vector_rank'],
                'vector_score': round(r['vector_score'], 4) if r['vector_score'] is not None else None,
                'bm25_rank': r['bm25_rank'],
                'bm25_score': round(r['bm25_score'], 3),
                'kb_hybrid_rank': r['kb_hybrid_rank'],
                'text_preview': r['text'][:150]
            }
            for r in results
        ]
    }
    if error:
        event['error'] = error
    elif not results:
        event['warning'] = 'No chunks retrieved'
    write_log_event(event)

    status = f"❌ {error}" if error else f"{len(results)} chunks in {latency_ms} ms"
    if rerank_info['error']:
        status += " (⚠️ reranking failed, hybrid order used)"
    elif rerank_info['reranked']:
        status += f" (reranked in {rerank_info['latency_ms']} ms)"
    print(f"📝 Logged retrieval {query_id}: {status}")
    return query_id, results


def show_retrieval_log(last_n: int = 3):
    """Print the most recent retrieval events with their chunks."""
    events = read_log_events('retrieval')[-last_n:]
    if not events:
        print("No retrieval events logged yet.")
        return
    for e in events:
        print(f"\n🗂️ {e['timestamp']}  query_id={e['query_id']}  ({e['latency_ms']} ms)")
        print(f"   Query: {e['query']}")
        if e.get('error') or e.get('warning'):
            print(f"   ⚠️ {e.get('error') or e.get('warning')}")
        if e.get('rerank_error'):
            print(f"   ⚠️ Reranking failed: {e['rerank_error'][:100]}")
        for c in e['chunks']:
            score = f"{c['rerank_score']:.3f}" if c.get('rerank_score') is not None else '-'
            print(f"   [{c['rank']}] p.{c['page'] or '-':<4} found by {c['found_by']:<8}"
                  f"hybrid rank {c.get('hybrid_rank') or '-'}, rerank score {score} "
                  f"| {c['text_preview'][:45].replace(chr(10), ' ')}...")


def retrieval_stats():
    """Summarize all logged retrievals."""
    events = read_log_events('retrieval')
    if not events:
        print("No retrieval events logged yet.")
        return {}

    latencies = [e['latency_ms'] for e in events]
    found_by = Counter(c['found_by'] for e in events for c in e['chunks'])
    files = {c['file'] for e in events for c in e['chunks']}
    pages = Counter((f"{c['file']} " if len(files) > 1 else '') + f"p.{c['page'] or '-'}"
                    for e in events for c in e['chunks'])
    failed = sum(1 for e in events if e.get('error') or e.get('warning'))
    reranked = [e for e in events if e.get('reranked')]
    rerank_failed = sum(1 for e in events if e.get('rerank_error'))
    promoted = sum(1 for e in reranked for c in e['chunks'] if (c.get('hybrid_rank') or 0) > e['k'])

    print("\n📊 RETRIEVAL MONITORING SUMMARY")
    print("=" * 80)
    print(f"Queries logged:           {len(events)}")
    print(f"Queries with no chunks:   {failed}")
    print(f"Average latency:          {sum(latencies) / len(latencies):.0f} ms (max {max(latencies)} ms)")
    print(f"Chunks found by both:     {found_by.get('both', 0)}")
    print(f"Chunks found by keyword:  {found_by.get('keyword', 0)}   (vector search alone would have missed these)")
    print(f"Chunks found by vector:   {found_by.get('vector', 0)}   (keyword search alone would have missed these)")
    print(f"Reranked queries:         {len(reranked)} (failed: {rerank_failed})")
    if reranked:
        print(f"Average rerank latency:   {sum(e['rerank_latency_ms'] for e in reranked) / len(reranked):.0f} ms")
        print(f"Moved in by reranker:     {promoted}   (chunks outside the hybrid top k that made the final top k)")
    print(f"Most retrieved pages:     {', '.join(f'{p} ({n}x)' for p, n in pages.most_common(5)) or '-'}")
    print("=" * 80)

    return {
        'queries': len(events),
        'queries_without_chunks': failed,
        'avg_latency_ms': sum(latencies) / len(latencies),
        'found_by': dict(found_by),
        'reranked_queries': len(reranked),
        'rerank_failures': rerank_failed,
        'top_pages': pages.most_common(5)
    }


# Test: run three monitored searches, then inspect the log
print("🧪 TESTING RETRIEVAL MONITORING\n")
for test_query in [
    "What are AnyCompany's operating lease liabilities?",
    "foreign exchange risk",
    "How does the company manage changes in commodity prices?"
]:
    monitored_search(test_query, k=3)

print(f"\n📁 Log file: {RAG_LOG_PATH}")
show_retrieval_log(last_n=3)
retrieval_stats()

### Code Explanation

**Why Monitor Retrieval:**  When a RAG answer is wrong, the cause is often the retrieval, not the model: the right chunk was never retrieved. Logging the retrieved chunks for every query shows exactly what the model was given.

**Log File:**  Events are appended to `logs/rag_log.jsonl`, next to this notebook, one JSON object per line (the JSON Lines format). The file can be appended to safely and loaded into pandas, Amazon Athena or Amazon CloudWatch Logs Insights. Step 7 writes `generation` events to the same file, linked by `query_id`.

**`monitored_search()`:**  Runs `search_and_rerank()` (hybrid search, then reranking) and logs a `retrieval` event with:
- `query_id`: a short random ID that links this retrieval to the answer generated from it.
- `timestamp` (UTC), `knowledge_base_id`, `query`, `method`, `k` and `latency_ms`.
- Reranking details: `reranked` (whether reranking succeeded), `rerank_model`, `rerank_latency_ms` and `rerank_error`.
- For every chunk: its final rank, its rank before reranking (`hybrid_rank`), its `rerank_score`, chunk ID, file, page, `found_by`, RRF score, vector rank and score, BM25 rank and score, its rank in Bedrock's `HYBRID` results, and the first 150 characters of its text.
- Pass `rerank=False` to log a search without reranking.
- An `error` field if the Retrieve call failed, or a `warning` field if no chunk was retrieved. A failed search is logged and returns no chunks instead of stopping the notebook.

**`show_retrieval_log()`:**  Prints the most recent retrievals with their chunks.

**`retrieval_stats()`:**  Summarizes all logged retrievals: average and maximum latency, how many chunks each method found on its own (this shows the value of hybrid search), how many queries were reranked and how many reranking calls failed, the average rerank latency, how many chunks the reranker moved into the final top `k`, and the most retrieved pages.

**Note:**  The log keeps growing each time you run the notebook. Delete `logs/rag_log.jsonl` to start with an empty log.

## Step 5: Citation Generation

Number the retrieved chunks as sources, make the model cite them after every fact, and return the source document references.

In [ ]:
# Step 5: Citation generation (return source document references with every answer)

ABSTAIN_MESSAGE = "I don't have enough information in the provided sources to answer this question."
GAP_PATTERN = re.compile(r"\bsources? (?:do|does) not (?:state|provide|contain|include|mention|specify)\b", re.IGNORECASE)
CITATION_PATTERN = re.compile(r'\[\s*(S\d+(?:\s*[,;]\s*S\d+)*)\s*\]', re.IGNORECASE)


def invoke_llm(prompt: str, model_id: str = LLM_MODEL_ID, max_tokens: int = 1000, temperature: float = 0.3) -> str:
    """Send one prompt to an Amazon Nova model and return the text reply."""
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        contentType='application/json',
        accept='application/json',
        body=json.dumps({
            "schemaVersion": "messages-v1",
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature, "topP": 0.9}
        })
    )
    response_body = json.loads(response['body'].read())
    return response_body['output']['message']['content'][0]['text']


def source_label(r: Dict) -> str:
    return f"{r['file']}, page {r['page']}" if r['page'] is not None else r['file']


def build_numbered_context(results: List[Dict]) -> str:
    """Label each retrieved chunk as [S1], [S2], ... with its file name and page."""
    return "\n\n".join(f"[S{n}] (source: {source_label(r)})\n{r['text']}" for n, r in enumerate(results, 1))


def generate_cited_answer(question: str, results: List[Dict]) -> str:
    """Answer the question from the numbered sources, citing a source ID after every fact."""
    if not results:
        return ABSTAIN_MESSAGE
    prompt = f"""You are a financial analyst AI assistant. Answer the question using ONLY the numbered excerpts from AnyCompany's financial statements below.

Sources:
{build_numbered_context(results)}

Question: {question}

Rules:
- Use only facts stated in the sources. Do not add outside knowledge.
- End every sentence or bullet point that states a fact with the source ID(s) it comes from, in square brackets, for example [S1] or [S1][S3].
- Use only source IDs that appear in the list above.
- Copy numbers, dates and currency amounts exactly as they appear in the sources. Do not round them or convert units.
- If the sources answer only part of the question, answer that part, and for each missing detail write one sentence that starts with "The sources do not state".
- If the sources do not contain the answer at all, reply with exactly: {ABSTAIN_MESSAGE}
- Be concise. Use bullet points for lists.

Answer:"""
    try:
        return invoke_llm(prompt)
    except Exception as e:
        return f"❌ Error generating answer: {explain_bedrock_error(e)}"


def normalize(text: str) -> str:
    return text.replace('’', "'").lower().strip()


def is_abstention(text: str) -> bool:
    return normalize(ABSTAIN_MESSAGE).rstrip('.') in normalize(text)


def split_into_statements(answer: str) -> List[tuple]:
    """Split an answer into sentences and bullet points, keeping each one's citations with it.
    Returns (sentence, line_number) pairs."""
    statements = []
    for line_number, line in enumerate(answer.splitlines()):
        line = line.strip().replace('**', '')
        line = re.sub(r'^(?:[-*•]|\d+[.)])\s+', '', line)        # remove bullet markers and list numbers
        if not line or line.startswith('#') or line.endswith(':'):
            continue                                              # skip blank lines, headings, "Key risks:"
        # Move citations that follow the full stop in front of it: "fact. [S1]" -> "fact [S1]."
        line = re.sub(r'([.!?])((?:\s*\[\s*S\d+[^\]]*\])+)', r'\2\1', line, flags=re.IGNORECASE)
        for sentence in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"(])', line):
            sentence = sentence.strip()
            if len(CITATION_PATTERN.sub('', sentence).split()) >= 3:
                statements.append((sentence, line_number))
    return statements


def extract_citations(answer: str, results: List[Dict], question: str = '') -> Dict:
    """Map every [S#] in the answer to the source document, page and chunk it refers to."""
    statements, gaps, cited, invalid = [], [], set(), set()

    for sentence, line_number in split_into_statements(answer):
        if is_abstention(sentence):
            continue
        clean = CITATION_PATTERN.sub('', sentence).replace(' .', '.').strip()
        if GAP_PATTERN.search(clean):          # "The sources do not state ..." is not a factual claim
            gaps.append(clean)
            continue
        source_ids = []
        for group in CITATION_PATTERN.findall(sentence):
            for sid in re.split(r'\s*[,;]\s*', group.upper()):
                n = int(sid[1:])
                if 1 <= n <= len(results):
                    source_ids.append(n)
                else:
                    invalid.add(sid)                              # e.g. [S7] when only 5 sources exist
        statements.append({'text': clean, 'source_ids': sorted(set(source_ids)), 'line': line_number})

    # A bullet or paragraph often cites its sources once, at the end. Sentences before
    # that citation, on the same line, inherit it.
    for i in range(len(statements) - 2, -1, -1):
        current, following = statements[i], statements[i + 1]
        if not current['source_ids'] and following['source_ids'] and current['line'] == following['line']:
            current['source_ids'] = following['source_ids']
    for s in statements:
        cited.update(s['source_ids'])

    references = []
    for n in sorted(cited):
        r = results[n - 1]
        references.append({
            'id': f'S{n}',
            'file': r['file'],
            'page': r['page'],
            's3_uri': r['uri'],
            'chunk_id': r['chunk_id'],
            'vector_score': r.get('vector_score'),
            'rerank_score': r.get('rerank_score'),
            'text_preview': r['text'][:200]
        })

    return {
        'question': question,
        'answer': answer,
        'abstained': is_abstention(answer) and not statements,
        'statements': statements,
        'missing_details': gaps,
        'references': references,
        'invalid_citations': sorted(invalid),
        'uncited_sources': [f'S{n}' for n in range(1, len(results) + 1) if n not in cited]
    }


def print_references(citation_info: Dict):
    """Print the source document references for an answer."""
    print("\n📖 REFERENCES:")
    if citation_info['abstained']:
        print("   (The model said the sources do not contain the answer, so nothing is cited.)")
        return
    if not citation_info['references']:
        print("   ⚠️ The answer does not cite any source.")
    for ref in citation_info['references']:
        page = f", page {ref['page']}" if ref['page'] is not None else ''
        if ref['rerank_score'] is not None:
            score = f"  (rerank score {ref['rerank_score']:.2f})"
        elif ref['vector_score'] is not None:
            score = f"  (relevance {ref['vector_score']:.2f})"
        else:
            score = ''
        print(f"   [{ref['id']}] {ref['file']}{page}{score}")
        print(f"        {ref['s3_uri']}")
    for gap in citation_info['missing_details']:
        print(f"   ℹ️ Not in sources: {gap}")
    if citation_info['invalid_citations']:
        print(f"   ⚠️ Cited IDs that do not exist: {', '.join(citation_info['invalid_citations'])}")
    if citation_info['uncited_sources']:
        print(f"   ℹ️ Retrieved but not cited: {', '.join(citation_info['uncited_sources'])}")


# Test: the lease question from above, from hybrid retrieval to a cited answer
print("=" * 90)
print("🧪 TESTING CITATION GENERATION")
print("=" * 90)

test_question = globals().get('query') or "What was the total operating lease liabilities and total sublease income of the AnyCompany as of December 31, 2022?"
query_id, test_results = monitored_search(test_question, k=5)
cited_answer = generate_cited_answer(test_question, test_results)
test_citations = extract_citations(cited_answer, test_results, test_question)

print(f"\n❓ {test_question}")
print("\n💬 ANSWER:")
print(cited_answer)
print_references(test_citations)

### Code Explanation

**`invoke_llm()`:**  One reusable function that sends a prompt to an Amazon Nova model and returns the reply. It uses the same `messages-v1` request format as the cells above. Steps 5 and 6 both use it.

**`build_numbered_context()`:**  Labels every retrieved chunk as `[S1]`, `[S2]`, ... with its file name and page number, so the model can refer to it.

**`generate_cited_answer()`:**  The prompt tells the model to:
- use only facts from the sources,
- end every factual sentence or bullet with the source ID(s), for example `[S1]` or `[S1][S3]`,
- copy numbers and amounts exactly, without rounding or converting units (Step 6 checks them),
- write `The sources do not state ...` for each detail that is missing, instead of guessing,
- reply with a fixed sentence (`ABSTAIN_MESSAGE`) when the sources do not contain the answer.

The temperature is `0.3` instead of `0.5` (cell above), because a factual, cited answer should vary less.

**`split_into_statements()`:**  Splits the answer into sentences and bullet points. It removes bullet markers, skips headings and lead-in lines that end with a colon, and keeps a citation that follows the full stop (`fact. [S1]`) with its sentence.

**`extract_citations()`:**  Finds every `[S#]` in each statement and returns:
- `statements`: each statement with the source numbers it cites. A sentence without a citation inherits the citation of the next sentence on the same line, because a bullet often cites its sources once, at the end.
- `references`: for every cited source, its ID, file name, page, S3 URI, chunk ID, vector and rerank scores and a text preview.
- `missing_details`: the `The sources do not state ...` sentences. They are not facts, so Step 6 does not check them.
- `invalid_citations`: IDs that do not exist, for example `[S7]` when only 5 sources were retrieved.
- `uncited_sources`: sources that were retrieved but not used.
- `abstained`: `True` when the model said the sources do not contain the answer.

**`print_references()`:**  Prints the references, for example `[S1] AnyCompany_financial_10K.pdf, page 52 (rerank score 0.87)` followed by the S3 URI of the document.

**Expected Output:**  An answer with `[S#]` markers after its sentences, followed by a **REFERENCES** list. The model decides which sources to cite, so your answer and references will differ slightly from run to run. The page number appears when the Knowledge Base stored it for the PDF chunk.

## Step 6: Hallucination Detection Checks

Check every generated answer in three ways and give it a risk level. The second test plants errors in an answer to show that the checks catch them.

In [ ]:
# Step 6: Hallucination detection checks for RAG outputs

print(f"🛡️ Setting up hallucination checks (judge model: {JUDGE_MODEL_ID})...\n")

# Numbers on their own only: digits that are part of a name such as 10-K or 10K are ignored
NUMBER_PATTERN = re.compile(r'(?<![A-Za-z\d.,])(?<![A-Za-z]-)\d+(?:[.,]\d+)*(?![A-Za-z\d])(?![.,]\d)(?!-[A-Za-z])')
URL_PATTERN = re.compile(r'https?://\S+')


def find_numbers(text: str) -> set:
    """All numbers in a text, without thousands separators: '$1,234.5' -> '1234.5'."""
    return {n.replace(',', '') for n in NUMBER_PATTERN.findall(URL_PATTERN.sub('', text))}


# ------------------------------------------------------------------------------
# Check 1: citation check - every factual statement must cite a valid source
# ------------------------------------------------------------------------------
def check_citations(citation_info: Dict) -> Dict:
    statements = citation_info['statements']
    uncited = [s['text'] for s in statements if not s['source_ids']]
    coverage = (len(statements) - len(uncited)) / len(statements) if statements else None
    return {
        'citation_coverage': coverage,
        'uncited_statements': uncited,
        'invalid_citations': citation_info['invalid_citations']
    }


# ------------------------------------------------------------------------------
# Check 2: value check - every number and URL in a statement must appear in the
# sources it cites (or in any retrieved source, if it cites none). Numbers that
# are in the question, such as the year asked about, are also allowed.
# ------------------------------------------------------------------------------
def check_values(citation_info: Dict, results: List[Dict]) -> List[Dict]:
    all_sources_text = " ".join(r['text'] for r in results)
    question_numbers = find_numbers(citation_info.get('question', ''))
    issues = []
    for s in citation_info['statements']:
        if s['source_ids']:
            source_text = " ".join(results[n - 1]['text'] for n in s['source_ids'])
        else:
            source_text = all_sources_text

        for url in URL_PATTERN.findall(s['text']):
            url = url.rstrip('.,;)')
            if url not in source_text:
                issues.append({'statement': s['text'], 'value': url, 'type': 'link'})

        allowed = find_numbers(source_text) | question_numbers
        for number in sorted(find_numbers(s['text'])):
            if number not in allowed:
                issues.append({'statement': s['text'], 'value': number, 'type': 'number'})
    return issues


# ------------------------------------------------------------------------------
# Check 3: LLM judge - a second model call checks every statement against the sources
# ------------------------------------------------------------------------------
def parse_json_object(text: str):
    """Extract the first JSON object from a model reply (which may be wrapped in ```json fences)."""
    start, end = text.find('{'), text.rfind('}')
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def judge_statements(citation_info: Dict, results: List[Dict]) -> Dict:
    statements = citation_info['statements']
    if not statements:
        return {'status': 'skipped', 'verdicts': []}

    numbered_statements = "\n".join(f"{i}. {s['text']}" for i, s in enumerate(statements, 1))
    prompt = f"""You are a strict fact-checker for a financial question-answering system.
Decide whether each numbered statement is supported by the sources below.

Sources:
{build_numbered_context(results)}

Statements:
{numbered_statements}

For each statement, give one verdict:
- SUPPORTED: the sources state it, or it follows directly from them
- NOT_SUPPORTED: any part of it, including any number, date or amount, is missing from the sources or contradicts them

Reply with JSON only, no other text, in exactly this format:
{{"results": [{{"id": 1, "verdict": "SUPPORTED", "reason": "short reason"}}]}}"""

    try:
        raw_reply = invoke_llm(prompt, model_id=JUDGE_MODEL_ID, max_tokens=2000, temperature=0.1)
    except Exception as e:
        return {'status': 'error', 'error': explain_bedrock_error(e), 'verdicts': []}

    parsed = parse_json_object(raw_reply)
    if not parsed or not isinstance(parsed.get('results'), list):
        return {'status': 'unparsed', 'raw_reply': raw_reply[:300], 'verdicts': []}

    verdicts, judged_ids = [], set()
    for item in parsed['results']:
        try:
            statement_id = int(item.get('id'))
        except (TypeError, ValueError):
            continue
        if not 1 <= statement_id <= len(statements) or statement_id in judged_ids:
            continue
        judged_ids.add(statement_id)
        verdict = str(item.get('verdict', '')).upper().replace(' ', '_')
        if verdict not in ('SUPPORTED', 'NOT_SUPPORTED'):
            verdict = 'UNCLEAR'
        verdicts.append({'statement': statements[statement_id - 1]['text'],
                         'verdict': verdict, 'reason': item.get('reason', '')})

    for i, s in enumerate(statements, 1):                         # statements the judge skipped
        if i not in judged_ids:
            verdicts.append({'statement': s['text'], 'verdict': 'UNCLEAR', 'reason': 'Not checked by the judge'})

    return {'status': 'ok', 'verdicts': verdicts}


# ------------------------------------------------------------------------------
# Combine the three checks into one report with a risk level
# ------------------------------------------------------------------------------
def detect_hallucinations(citation_info: Dict, results: List[Dict], use_judge: bool = True) -> Dict:
    if citation_info['answer'].startswith('❌'):
        return {'risk': 'ERROR', 'reasons': ['The answer could not be generated'], 'faithfulness': None}

    if citation_info['abstained']:
        return {'risk': 'ABSTAINED', 'reasons': ['The model said the sources do not contain the answer'],
                'faithfulness': None, 'citation_coverage': None, 'uncited_statements': [],
                'invalid_citations': [], 'value_issues': [], 'judge': {'status': 'skipped', 'verdicts': []}}

    citations = check_citations(citation_info)
    value_issues = check_values(citation_info, results)
    judge = judge_statements(citation_info, results) if use_judge else {'status': 'skipped', 'verdicts': []}

    verdicts = judge['verdicts']
    unsupported = [v for v in verdicts if v['verdict'] == 'NOT_SUPPORTED']
    unclear = [v for v in verdicts if v['verdict'] == 'UNCLEAR']
    supported = [v for v in verdicts if v['verdict'] == 'SUPPORTED']
    faithfulness = len(supported) / len(verdicts) if verdicts else None

    reasons = []
    if unsupported:
        reasons.append(f"{len(unsupported)} statement(s) not supported by the sources (judge)")
    if value_issues:
        reasons.append(f"{len(value_issues)} number(s)/link(s) not found in the cited sources")
    if citations['invalid_citations']:
        reasons.append(f"Cites sources that do not exist: {', '.join(citations['invalid_citations'])}")
    high = bool(reasons)

    if not citation_info['statements']:
        reasons.append("The answer contains no factual statements to check")
    if citations['uncited_statements']:
        reasons.append(f"{len(citations['uncited_statements'])} statement(s) without a citation")
    if unclear:
        reasons.append(f"{len(unclear)} statement(s) the judge could not classify")
    if use_judge and judge['status'] not in ('ok', 'skipped'):
        reasons.append(f"Judge check did not complete (status: {judge['status']})")

    risk = 'HIGH' if high else ('MEDIUM' if reasons else 'LOW')
    return {
        'risk': risk,
        'reasons': reasons or ['All statements are cited, values match the sources, and the judge found them supported'],
        'faithfulness': faithfulness,
        'citation_coverage': citations['citation_coverage'],
        'uncited_statements': citations['uncited_statements'],
        'invalid_citations': citations['invalid_citations'],
        'value_issues': value_issues,
        'judge': judge
    }


RISK_ICONS = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴', 'ABSTAINED': '⚪', 'ERROR': '❌'}


def print_hallucination_report(report: Dict):
    label = report['risk'] if report['risk'] in ('ABSTAINED', 'ERROR') else f"{report['risk']} RISK"
    print(f"\n🛡️ HALLUCINATION CHECK: {RISK_ICONS.get(report['risk'], '')} {label}")
    for reason in report['reasons']:
        print(f"   • {reason}")
    if report.get('faithfulness') is not None:
        print(f"   Faithfulness (judge):  {report['faithfulness']:.0%} of statements supported")
    if report.get('citation_coverage') is not None:
        print(f"   Citation coverage:     {report['citation_coverage']:.0%} of statements cited")
    for issue in report.get('value_issues', []):
        print(f"   🔢 {issue['type']} '{issue['value']}' not in cited source → \"{issue['statement'][:70]}...\"")
    for statement in report.get('uncited_statements', []):
        print(f"   📎 No citation → \"{statement[:80]}\"")
    for v in report.get('judge', {}).get('verdicts', []):
        if v['verdict'] != 'SUPPORTED':
            print(f"   ⚖️ {v['verdict']} → \"{v['statement'][:70]}\" ({v['reason']})")
    if report.get('judge', {}).get('status') == 'error':
        print(f"   ❌ Judge error: {report['judge']['error']}")
    if report.get('judge', {}).get('status') == 'unparsed':
        print(f"   ⚠️ Judge reply was not valid JSON: {report['judge']['raw_reply'][:120]}")


# Test 1: the real answer generated in Step 5
print("=" * 90)
print("🧪 TEST 1: Check the answer generated in Step 5")
print("=" * 90)
print(f"❓ {test_question}")
report = detect_hallucinations(test_citations, test_results)
print_hallucination_report(report)

# Test 2: an answer with planted errors, to show that the checks catch them
print("\n" + "=" * 90)
print("🧪 TEST 2: Check an answer with planted hallucinations")
print("=" * 90)
fabricated_answer = (
    "AnyCompany's total operating lease liabilities were $9,876 million as of December 31, 2022 [S1]. "
    "Its total sublease income was $777 million [S2]. "
    "AnyCompany plans to close all of its leased offices next year."
)
print(f"💬 {fabricated_answer}")
fabricated_citations = extract_citations(fabricated_answer, test_results, test_question)
print_hallucination_report(detect_hallucinations(fabricated_citations, test_results))

### Code Explanation

A RAG answer **hallucinates** when it states something that is not in the retrieved sources. No single check catches every case, so this step combines three.

**Check 1 - Citations (`check_citations()`):**  Every factual statement should cite a source. It reports `citation_coverage` (the share of statements that cite a source), the statements without a citation, and any citations to sources that do not exist.

**Check 2 - Numbers and Links (`check_values()`):**  Every number and URL in a statement must appear in the sources that statement cites (or in any retrieved source, if it cites none). Invented figures and dates are the most harmful hallucinations in a financial answer, and this check catches them without a model call. It also catches a correct number cited to the wrong source.
- Thousands separators are ignored, so `$1,234` matches `1234`.
- Numbers in the question, such as the year asked about, are allowed.
- Digits that are part of a name, such as `10-K`, are not treated as numbers.

**Check 3 - LLM Judge (`judge_statements()`):**  A second model call receives the numbered sources and the numbered statements and returns `SUPPORTED` or `NOT_SUPPORTED` for each one, as JSON. This catches unsupported claims that contain no numbers. `parse_json_object()` extracts the JSON even if the model wraps it in ```` ```json ```` fences. A reply that cannot be parsed is reported, not ignored. The temperature is `0.1`, so the verdicts are as consistent as possible. The judge uses `JUDGE_MODEL_ID` from Step 1.

**`detect_hallucinations()`:**  Combines the checks into one risk level:

| Risk | When |
|---|---|
| 🔴 **HIGH** | The judge found an unsupported statement, a number or link is not in its cited source, or the answer cites a source that does not exist. |
| 🟡 **MEDIUM** | Nothing was found to be wrong, but some statements have no citation, or the judge could not check every statement. |
| 🟢 **LOW** | All statements are cited, all values match the sources, and the judge found every statement supported. |
| ⚪ **ABSTAINED** | The model said the sources do not contain the answer. This is the correct behaviour for an out-of-scope question, not a hallucination. |

It also reports **faithfulness**: the share of statements the judge found supported.

**Test 1:**  Checks the real answer from Step 5. A well-grounded answer is usually LOW risk. If it is MEDIUM or HIGH, read the listed statements: often the model added a summary sentence without a citation, or rounded a figure.

**Test 2:**  Checks an answer with three planted errors. Expect **HIGH** risk, with `9876` and `777` reported as numbers not in the cited source, the last sentence reported as having no citation, and the judge marking the invented statements `NOT_SUPPORTED`. If fewer than two chunks were retrieved, `[S2]` is also reported as a source that does not exist.

**Limits:**  These checks reduce hallucinations; they do not prove an answer correct. The judge is itself a model and can be wrong, and the number check cannot tell whether a number is used in the right meaning. Review HIGH-risk answers before you use them.

**Cost:**  The judge adds one model call per checked answer. Pass `use_judge=False` to `detect_hallucinations()` to run only the free checks 1 and 2.

## Step 7: Complete Pipeline with Reranking, Citations and Hallucination Checks

Run hybrid retrieval, reranking, monitoring, cited answer generation and hallucination checks together, then summarize the log.

In [ ]:
# Step 7: Complete pipeline with hybrid search, reranking, monitoring, citations and hallucination checks

def ask_with_citations(question: str, k: int = 5, use_judge: bool = True, rerank: bool = None,
                       verbose: bool = True) -> Dict:
    """Retrieve, rerank, answer with citations, check for hallucinations and log everything."""
    # 1. Hybrid retrieval + reranking, logged as a 'retrieval' event (Steps 2, 3 and 4)
    query_id, results = monitored_search(question, k=k, rerank=rerank)

    # 2. Cited answer and source references (Step 5)
    start = time.time()
    answer = generate_cited_answer(question, results)
    generation_ms = round((time.time() - start) * 1000)
    citation_info = extract_citations(answer, results, question)

    # 3. Hallucination checks (Step 6)
    start = time.time()
    report = detect_hallucinations(citation_info, results, use_judge=use_judge)
    check_ms = round((time.time() - start) * 1000)

    # 4. Log a 'generation' event with the same query_id as the retrieval
    write_log_event({
        'event': 'generation',
        'query_id': query_id,
        'timestamp': utc_now(),
        'question': question,
        'llm_model': LLM_MODEL_ID,
        'judge_model': JUDGE_MODEL_ID if use_judge else None,
        'answer': answer,
        'references': [{key: ref[key] for key in ('id', 'file', 'page', 's3_uri', 'chunk_id', 'rerank_score')}
                       for ref in citation_info['references']],
        'missing_details': citation_info['missing_details'],
        'risk': report['risk'],
        'reasons': report['reasons'],
        'faithfulness': report.get('faithfulness'),
        'citation_coverage': report.get('citation_coverage'),
        'generation_latency_ms': generation_ms,
        'check_latency_ms': check_ms
    })

    if verbose:
        print(f"\n❓ {question}")
        print("\n💬 ANSWER:")
        print(answer)
        print_references(citation_info)
        print_hallucination_report(report)

    return {
        'query_id': query_id,
        'question': question,
        'answer': answer,
        'references': citation_info['references'],
        'hallucination_report': report,
        'sources': results
    }


def quality_summary():
    """Summarize every logged answer."""
    events = read_log_events('generation')
    if not events:
        print("No answers logged yet.")
        return {}

    risks = Counter(e['risk'] for e in events)
    faithfulness = [e['faithfulness'] for e in events if e.get('faithfulness') is not None]
    coverage = [e['citation_coverage'] for e in events if e.get('citation_coverage') is not None]
    high_risk = [e['query_id'] for e in events if e['risk'] == 'HIGH']

    print("\n📊 ANSWER QUALITY SUMMARY")
    print("=" * 80)
    print(f"Answers logged:          {len(events)}")
    for risk in ('LOW', 'MEDIUM', 'HIGH', 'ABSTAINED', 'ERROR'):
        print(f"  {RISK_ICONS[risk]} {risk:<10}           {risks.get(risk, 0)}")
    if faithfulness:
        print(f"Average faithfulness:    {sum(faithfulness) / len(faithfulness):.0%}")
    if coverage:
        print(f"Average citation cover.: {sum(coverage) / len(coverage):.0%}")
    if high_risk:
        print(f"HIGH-risk answers to review (query_id): {', '.join(high_risk[-5:])}")
    print("=" * 80)
    return {'answers': len(events), 'risk_counts': dict(risks), 'high_risk_query_ids': high_risk}


# Test: the two sample questions from the activity guide and one question the document cannot answer
pipeline_questions = [
    "What are the key risks and uncertainties that could impact AnyCompany Financial's financial performance and position?",
    "How does AnyCompany Financial mitigate the risks associated with changes in commodity prices, foreign exchange rates, and equity prices?",
    "Which football team won the 2022 FIFA World Cup?"   # out of scope: expect ABSTAINED
]
for question in pipeline_questions:
    print("\n" + "=" * 90)
    ask_with_citations(question)

print()
retrieval_stats()
quality_summary()

### Code Explanation

**`ask_with_citations()`:**  Runs the full pipeline for one question:
1. `monitored_search()` retrieves the chunks with BM25 + vector search, reranks them with Cohere Rerank 3.5 and logs a `retrieval` event (Steps 2, 3 and 4). Pass `rerank=False` to skip reranking for one question.
2. `generate_cited_answer()` writes an answer with `[S#]` citations, and `extract_citations()` turns them into references (Step 5).
3. `detect_hallucinations()` runs the three checks (Step 6).
4. A `generation` event is written to the same log with the question, the models used, the answer, its references (file, page, S3 URI, chunk ID), the missing details, the risk level and reasons, faithfulness, citation coverage and latencies. Its `query_id` matches the retrieval event, so for any flagged answer you can look up exactly which chunks the model was given.

It returns a dictionary with the answer, references, hallucination report and sources, so it can be used behind an API or a web UI.

**`quality_summary()`:**  Summarizes every logged answer: how many had each risk level, the average faithfulness and citation coverage, and the `query_id` of recent HIGH-risk answers to review.

**Test Questions:**
- The two sample questions from the activity guide, about key risks and about how AnyCompany mitigates commodity, foreign exchange and equity price risk. Expect cited answers with LOW or MEDIUM risk.
- `Which football team won the 2022 FIFA World Cup?` The financial statements do not contain this, so expect ⚪ **ABSTAINED**. If the model answers from its own knowledge instead, the checks flag the answer, because it is not in any source.

**Cost:**  Each question makes two Retrieve calls, one Rerank call and two model calls (answer and judge). The reranker is billed per query; see Amazon Bedrock pricing for the current rate.